# 🔢 Lesson 07: Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/07_embeddings.ipynb)

> **Computers can't compare *meanings*. They can compare numbers. An embedding turns meaning into numbers.**

---

## 🎯 Goal
By the end of this lesson you can turn text into vectors, measure how similar two texts are, and find the chunk that best matches a question.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 06](06_chunking.ipynb) |
| 🔑 **API key** | Not needed (the model runs locally) |
| 🛠 **Libraries** | `sentence-transformers`, `numpy`, `pdfplumber`, `langchain-core` |

In [ ]:
!pip install -q -U sentence-transformers numpy pdfplumber langchain-core

In [1]:
from pathlib import Path
import urllib.request
import re
import unicodedata
from collections import Counter
import pdfplumber
from langchain_core.documents import Document

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"

pdf_path = next((p for p in [Path("data/pdf1.pdf"), Path("../data/pdf1.pdf")] if p.exists()), None)
if pdf_path is None:
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PDF_URL, pdf_path)

# Lesson 03: parse into sections
lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        for line in page.extract_text_lines():
            size = round(max(char["size"] for char in line["chars"]), 1)
            lines.append({"text": line["text"], "size": size, "page": page_number})

body_size = Counter(line["size"] for line in lines).most_common(1)[0][0]

sections = []
current = None
for line in lines:
    if line["size"] > body_size:
        current = {"title": line["text"], "page": line["page"], "text": []}
        sections.append(current)
    elif current is not None:
        current["text"].append(line["text"])
sections = [section for section in sections if section["text"]]

# Lesson 04: clean
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("(cid:127)", "\n•")
    text = re.sub(r"Full[n■]time", "Full-time", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

# Lessons 05-06: one chunk per section, with metadata
chunks = [
    Document(
        page_content=clean_text(" ".join(section["text"])),
        metadata={"source": pdf_path.name, "page": section["page"], "section": section["title"]},
    )
    for section in sections
]

print("chunks:", len(chunks))
print([chunk.metadata["section"] for chunk in chunks])

chunks: 4
['Non-Disclosure Agreement (NDA) Policy', 'Workplace Conduct Policy', 'Paid Time Off (PTO) Policy', 'Travel & Expense Policy']


## 💡 What is an embedding?

An **embedding model** reads a text and outputs a list of numbers: a **vector**.

- All texts get a vector of the **same length** (the model's *dimension*, for example 384 numbers).
- Texts with **similar meaning** get vectors that point in a **similar direction**.
- Texts with different meaning get vectors that point in different directions.

So "similar meaning" becomes a math question: *how close are these two vectors?* The most common measure is **cosine similarity**: the angle between two vectors. Close to `1` means very similar, near `0` means unrelated.

The numbers themselves mean nothing to a human. Only the **distances between vectors** matter.

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

vector = embedder.encode(["Full-time employees accrue PTO."])[0]
print("shape:", vector.shape)
print("first 5 numbers:", vector[:5])
print("max tokens the model reads:", embedder.max_seq_length)

d:\AI Source\Rag_torurial\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2064.04it/s]


shape: (384,)
first 5 numbers: [-0.04740979 -0.00108254  0.00998406 -0.00675151 -0.05062842]
max tokens the model reads: 256


## 🧮 Step 1: Embed all chunks

We pass the list of chunk texts to `encode` once. We also ask for **normalized** vectors (each scaled to length 1). This makes the comparison in the next step a simple dot product.

We do this **once**, ahead of time. Only the question is embedded each time someone asks.

In [3]:
chunk_texts = [chunk.page_content for chunk in chunks]
chunk_vectors = embedder.encode(chunk_texts, normalize_embeddings=True)

print("shape:", chunk_vectors.shape)
print("length of first vector:", round(float(np.linalg.norm(chunk_vectors[0])), 3))
print("longest chunk (characters):", max(len(text) for text in chunk_texts))

shape: (4, 384)
length of first vector: 1.0
longest chunk (characters): 558


## 🔍 Step 2: Search by meaning

To search, we:
1. embed the **question** with the same model,
2. compare it with every chunk vector,
3. keep the `top_k` highest scores.

For normalized vectors, **cosine similarity = dot product** (`@` in numpy). Higher means closer in meaning.

> ⚠️ Always use the **same model** for chunks and questions. Vectors from different models live in different "spaces" and can't be compared.

In [4]:
def search(question, top_k=2):
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ question_vector
    best = np.argsort(scores)[::-1][:top_k]
    return [(float(scores[i]), chunks[i]) for i in best]

for score, chunk in search("How many vacation days do I get after two years?", top_k=4):
    print(f'{score:.3f} | {chunk.metadata["section"]}')

0.401 | Paid Time Off (PTO) Policy
0.188 | Travel & Expense Policy
-0.045 | Non-Disclosure Agreement (NDA) Policy
-0.060 | Workplace Conduct Policy


## 🧠 Meaning, not keywords

The question above says **"vacation"**. The matching section says **"PTO"** and **"Paid Time Off"**, but the word "vacation" appears only *once*, buried in the text. A keyword search would struggle. Let's test with questions that share **no important word** with the answer:

In [5]:
questions = [
    "Can I work from home?",
    "What happens if someone harasses a coworker?",
    "Who pays for my flight to a client meeting?",
]

for question in questions:
    score, chunk = search(question, top_k=1)[0]
    print(f'{question}\n   -> {chunk.metadata["section"]} ({score:.3f})\n')

Can I work from home?
   -> Non-Disclosure Agreement (NDA) Policy (0.188)

What happens if someone harasses a coworker?
   -> Workplace Conduct Policy (0.452)

Who pays for my flight to a client meeting?
   -> Travel & Expense Policy (0.465)



### 👀 What we saw

| Question | Best chunk | Score |
|---|---|---|
| How many vacation days do I get after two years? | Paid Time Off (PTO) | 0.401 |
| What happens if someone harasses a coworker? | Workplace Conduct | 0.452 |
| Who pays for my flight to a client meeting? | Travel & Expense | 0.465 |
| Can I work from home? | NDA (**wrong**: no such section exists) | 0.188 |

Two lessons:
1. **Meaning beats keywords.** "flight" found "Travel & Expense" and "harasses" found "Workplace Conduct" without sharing the key words.
2. **Search always returns *something*.** For "work from home" there is no right answer in our document, yet we got a chunk. The only warning sign is the **low score**. A real system must decide what to do with low scores (for example, say "I don't know").

> ℹ️ These scores come from one model and four chunks. Don't treat `0.3` or any other number as a universal threshold. Choose one by measuring on your own data (Evaluation lessons).

## ⚠️ Things to watch out for

- **Long chunks get cut silently.** Our model reads at most **256 tokens**. Anything after that is ignored, with no error. Our chunks are short enough, but a long chunk would be embedded using only its beginning.
- **One model for chunks and questions.** Vectors from different models can't be compared.
- **Language matters.** `all-MiniLM-L6-v2` is built for English. Persian text needs a multilingual model (Lesson 08).
- **Embeddings are not understanding.** A question with a negation ("who is *not* allowed to...") can still match the chunk about the allowed case, because the topic is the same.
- **Embeddings are stored, not recomputed.** Right now `chunk_vectors` lives in a Python variable and disappears when the notebook closes. Storing them properly is the job of a **vector database** (Lesson 09).

## 📌 Summary

- An **embedding model** turns text into a **vector** (here, 384 numbers). Similar meaning gives vectors that point in a similar direction.
- **Cosine similarity** measures that closeness. With normalized vectors it is just a dot product (`chunk_vectors @ question_vector`).
- To search: embed the question with the **same model**, score every chunk, keep the `top_k`.
- Search matches **meaning**, not keywords: "flight" found "Travel & Expense".
- Search **always returns the nearest chunk**, even when nothing is relevant. Only the score tells you.
- Models have a **maximum input length** (256 tokens here) and a **language** they are good at.

## ✍️ Exercise (10 minutes)

1. Ask a question that is **not** covered by the handbook, for example *"What is the dental insurance policy?"*, with `search(question, top_k=1)`. What is the best score?
2. Ask two questions that **are** covered, in your own words. What are their scores?
3. Compare. Could a simple rule such as "if the best score is below X, answer 'I don't know'" work on your examples? What X would you pick, and how sure are you?

**Success:** you can explain in one sentence why choosing X from only a few examples is risky.

💡 Solution: `solutions/07_embeddings.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Test the **256-token limit**. Build a long text by repeating a boring sentence many times, then add an important sentence at the very end, for example *"The secret code is 4711."*

1. Embed this long text and compare it with the question *"What is the secret code?"*.
2. Now embed *only* the last sentence and compare it with the same question.

Which score is higher? What does that tell you about **long chunks**?

## 📖 Next

We used an English model, and it worked for English. But the world (and our audience) writes in many languages. In the next lesson we compare embedding models, including multilingual ones.

**➡️ Next lesson:** [08 Choosing an embedding model](08_choosing_embedding_model.ipynb)